# 01. Предобработка журнала событий СМВУ, 2019-2026

## Что за данные и зачем их готовить

Москоллектор передал нам журнал событий системы мониторинга и управления (СМВУ) инженерных коллекторов.
Каждая строка журнала – одно сообщение от одного канала датчика: газоанализатор прислал очередной замер,
дверь открылась, насос включился, дымовой датчик перешел в «Неисправен». Журнал разбит на восемь годовых
CSV-файлов (2019 – первая половина 2026 года), вместе около 16 GB и примерно 313 млн строк. К нему приложены
справочники: каналы датчиков (тип датчика, название, объект), дерево объектов и словарь состояний, а также
пример журнала за один день, 2026-08-01, в том виде, в котором система будет получать данные в работе.

Прежде чем искать в журнале предвестники инцидентов, нужно убедиться, что сам журнал читается однозначно:
в нем нет повторов, которые завысят счетчики, нет «мусорных» строк, флаг тревоги понимается одинаково во всех
годах, а значения датчиков разобраны на числа и состояния. Этот ноутбук делает ровно это: один раз читает
все CSV, находит в них особенности формата, проверяет каждую на данных, очищает журнал и сохраняет его
в компактное хранилище Parquet. Все дальнейшие исследования работают только с этим хранилищем.

Про содержание мы на этом этапе ничего не выбрасываем. По словам заказчика, в 2021 году внедрялась новая
СМВУ и какое-то время старая и новая системы работали параллельно. Как это отразилось на журнале, будем
разбираться при его исследовании, а здесь сохраняем все периоды целиком и убираем только то, что
является дефектом выгрузки, а не событием.

*Техническое примечание.* Для пробного прогона годы можно ограничить переменной окружения
`ETL_YEARS` (например, `ETL_YEARS=2019`). Все цифры в тексте относятся к полному прогону по всем годам.

In [1]:
import os
import shutil
import time

import pandas as pd
from pyspark.sql import functions as F

from annanas import etl
from annanas.paths import DATASETS_DIR, EVENTS_DIR, PROCESSED_DIR
from annanas.spark import get_spark

YEARS = [int(y) for y in os.environ.get("ETL_YEARS", "2019,2020,2021,2022,2023,2024,2025,2026").split(",")]
JOURNALS = {year: DATASETS_DIR / f"ext-journal-{year}.csv" for year in YEARS}
SAMPLE = DATASETS_DIR / "журнал_событий_пример.csv"
print("Годы:", YEARS)

spark = get_spark("01_preprocessing")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 80)

Годы: [2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/24 19:01:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/24 19:01:08 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


In [2]:
files = pd.DataFrame(
    [{"файл": p.name, "размер_гб": round(p.stat().st_size / 2**30, 2)} for p in [*JOURNALS.values(), SAMPLE]]
    + [{"файл": name, "размер_гб": round((DATASETS_DIR / name).stat().st_size / 2**30, 4)}
       for name in ["справочник_каналов_датчиков.csv", "справочник_объектов_диспетчер.csv",
                    "справочник_состояний.csv"]]
)
files

,файл,размер_гб
0,ext-journal-2019.csv,1.1000
1,ext-journal-2020.csv,1.4700
2,ext-journal-2021.csv,2.5500
3,ext-journal-2022.csv,2.0100
4,ext-journal-2023.csv,1.5200
5,ext-journal-2024.csv,2.2500
6,ext-journal-2025.csv,2.5200
7,ext-journal-2026.csv,1.3900
8,журнал_событий_пример.csv,0.0100
9,справочник_каналов_датчиков.csv,0.0012


В каждом журнале шесть столбцов: `ид_события`, `ид_канала_данных`, `дата`, `время` (московское),
`тревожное` (флаг тревоги, который выставила сама СМВУ) и `значение_датчика`. Так выглядят первые строки
примера за 2026-08-01:

In [3]:
sample_raw = pd.read_csv(SAMPLE, dtype=str)
sample_raw.head()

,ид_события,ид_канала_данных,дата,время,тревожное,значение_датчика
0,4524243389,120473,2026-08-01,03:09:27,false,28
1,4524253385,120475,2026-08-01,03:19:55,false,29
2,4524594432,120466,2026-08-01,09:11:40,false,25
3,4524782574,120298,2026-08-01,12:17:17,false,28
4,4524806333,120511,2026-08-01,12:41:01,false,28


## 1. Справочники каналов и объектов

Журнал ссылается на канал только номером `ид_канала_данных`. Чтобы понимать, что это за датчик и где он
стоит, к каналу нужно подтянуть справочник каналов (тип датчика, название, объект) и дерево объектов.
Посмотрим на оба справочника.

In [4]:
channels_raw = pd.read_csv(DATASETS_DIR / "справочник_каналов_датчиков.csv")
objects_raw = pd.read_csv(DATASETS_DIR / "справочник_объектов_диспетчер.csv")
print("каналов:", len(channels_raw), "| уникальных ид_канала_данных:", channels_raw["ид_канала_данных"].nunique())
print("объектов:", len(objects_raw))
channels_raw.head(3)

каналов: 11485 | уникальных ид_канала_данных: 11485
объектов: 95


,ид_канала_данных,тип_инж_системы,тип_датчика,тег_инженерной_системы,название_датчика,ид_объект
0,120578,Охранная подсистема,КД АВ,15-11.1.131.2.,КД АВ ПК28,20
1,120504,Пожарная охрана,Тепловой датчик,15-11.1.104.2.,ТД ПК86-85,20
2,120298,Температурная подсистема,Датчик температуры,15-11.1.4.1.,"Темп. ВШ ПК88,5",20


In [5]:
levels = ["иерархия_уровень", "вид_объекта"]
(objects_raw.groupby(levels).size().rename("количество_объектов").to_frame()
 .join(channels_raw.merge(objects_raw, on="ид_объект").groupby(levels).size().rename("каналов"))
 .fillna(0).astype(int).reset_index())

,иерархия_уровень,вид_объекта,количество_объектов,каналов
0,1,district,1,0
1,2,controlHouse,16,0
2,3,controlHouse,49,3501
3,3,guardObject,29,7984


Объекты образуют дерево из трех уровней: наверху один район по эксплуатации, под ним 16 объектов второго
уровня, а под ними 78 объектов третьего уровня. Все 11 485 каналов привязаны к объектам третьего уровня,
поэтому объект третьего уровня – естественная единица, для которой будем строить прогноз, а его родитель
второго уровня дает группировку крупнее.

Координат у датчиков нет. Но в названии канала часто стоит пикет – отметка расстояния вдоль коллектора,
например `ГАЗ Д26 ПК746` или `Темп. ВШ ПК88,5`. Это единственная доступная локация внутри объекта, поэтому
вытаскиваем номер пикета из названия. Формат записи неоднороден: встречаются нижний регистр, пробел после
«ПК», дробная часть через запятую и диапазоны вида `ПК86-85` (берем первое число).

In [6]:
names = channels_raw["название_датчика"]
pk_examples = pd.concat([
    names[names.str.contains(r"ПК\d+,\d", regex=True)].head(2),
    names[names.str.contains(r"пк", regex=False)].head(2),
    names[names.str.contains(r"ПК\d+-\d", regex=True)].head(2),
    names[~names.str.contains(r"пк", case=False)].head(3),
])
pd.DataFrame({"название_датчика": pk_examples, "пикет": pk_examples.map(etl.extract_pk)})

,название_датчика,пикет
2,"Темп. ВШ ПК88,5",88.5
5,"ОД ВШ ПК88,5",88.5
916,ОД пк 765 4х секц.люк,765.0
2940,ГАЗ Д10 пк212,212.0
1,ТД ПК86-85,86.0
8,ТД ПК29-30,29.0
7,[Охранная зона]0: 0: Гагаринский тоннель,NaN
313,Охранная зона Ленинский 83,NaN
384,ОС ЦДТ,NaN


Функция `etl.build_channels` объединяет справочник каналов с деревом объектов (объект канала и название его
родителя) и добавляет столбец `пикет`. Столбец `тег_инженерной_системы` удаляем: это уникальный технический
код канала, организаторы разрешили его не использовать.

In [7]:
channels = etl.build_channels(channels_raw, objects_raw)
print("каналов без объекта:", channels["диспетчерское_название_объекта"].isna().sum())
print("доля каналов с пикетом:", round(channels["пикет"].notna().mean(), 3))
(channels.assign(без_пикета=channels["пикет"].isna())
 .groupby("тип_датчика")["без_пикета"].agg(каналов="size", каналов_без_пикета="sum", доля_без_пикета="mean")
 .round(3).sort_values("каналов_без_пикета", ascending=False))

каналов без объекта: 0
доля каналов с пикетом: 0.933


,каналов,каналов_без_пикета,доля_без_пикета
тип_датчика,,,
КД Дверь,917,156,0.170
Состояние фазы,957,123,0.129
Датчик дыма,4337,110,0.025
Переключатель,801,93,0.116
Датчик движения,804,80,0.100
Состояние охраны,47,44,0.936
Датчик температуры,600,41,0.068
Состояние насоса,180,33,0.183
Стекло,29,29,1.000


In [8]:
channels.to_parquet(PROCESSED_DIR / "channels.parquet", index=False)
channels.head(3)

,ид_канала_данных,тип_инж_системы,тип_датчика,название_датчика,ид_объект,иерархия_уровень,вид_объекта,диспетчерское_название_объекта,родитель,название_родителя,пикет
0,120578,Охранная подсистема,КД АВ,КД АВ ПК28,20,3,guardObject,объект Фита,6,объект Бета,28.0
1,120504,Пожарная охрана,Тепловой датчик,ТД ПК86-85,20,3,guardObject,объект Фита,6,объект Бета,86.0
2,120298,Температурная подсистема,Датчик температуры,"Темп. ВШ ПК88,5",20,3,guardObject,объект Фита,6,объект Бета,88.5


**Итог по справочникам.** У всех 11 485 каналов нашелся объект, и все эти объекты третьего уровня.
Пикет удалось извлечь у 93,3 % каналов. Без пикета осталось около 770 каналов, больше всего среди дверей,
фаз питания, дымовых датчиков, переключателей и датчиков движения; целиком без пикета датчики «Стекло»
и почти целиком «Состояние охраны». Судя по названиям без пикета («Охранная зона Ленинский 83», «ОС ЦДТ»),
это оборудование, привязанное не к точке трассы, а к объекту или его зоне целиком; для таких каналов
локацией будет сам объект.

## 2. Первый осмотр сырых журналов

Годовые файлы слишком велики для pandas, поэтому читаем их через Spark. Все столбцы читаем как строки:
если сразу попросить Spark привести `дата` к дате, а `тревожное` к логическому типу, он молча превратит
все, что не распознал, в пустые значения, и мы не увидим, что именно там было.

За один проход по всем файлам (и заодно по примеру за 2026-08-01) для каждого файла считаем:
сколько в нем строк; какие значения встречаются во флаге `тревожное`; сколько строк, где `дата` или `время`
не похожи на дату и время, и какие это строки; и на что похоже `значение_датчика` – на число, на дату
или на текст.

In [9]:
DATE_RE, TIME_RE = r"^\d{4}-\d{2}-\d{2}$", r"^\d{2}:\d{2}:\d{2}$"
raw_all = (spark.read.option("header", True).schema(etl.RAW_SCHEMA)
           .csv([str(p) for p in [*JOURNALS.values(), SAMPLE]])
           .withColumn("файл", F.col("_metadata.file_name")))
value = F.col("значение_датчика")
is_number = F.expr("try_cast(`значение_датчика` as double)").isNotNull()
is_date_value = value.rlike(etl.DATETIME_VALUE_RE)
odd_datetime = ~F.col("дата").rlike(DATE_RE) | ~F.col("время").rlike(TIME_RE)

first_look = (
    raw_all.groupBy("файл").agg(
        F.count("*").alias("строк"),
        F.array_sort(F.collect_set("тревожное")).alias("значения_флага"),
        F.count(F.when(odd_datetime, 1)).alias("строк_с_нераспознанной_датой"),
        F.slice(F.collect_list(F.when(odd_datetime, F.concat_ws(" | ", *etl.RAW_COLUMNS))), 1, 3)
         .alias("примеры_таких_строк"),
        F.count(F.when(is_number, 1)).alias("значение_число"),
        F.count(F.when(is_date_value, 1)).alias("значение_дата"),
        F.count(F.when(~is_number & ~is_date_value, 1)).alias("значение_текст"),
    )
    .orderBy("файл").toPandas()
)
first_look

,файл,строк,значения_флага,строк_с_нераспознанной_датой,примеры_таких_строк,значение_число,значение_дата,значение_текст
0,ext-journal-2019.csv,20993328,"[f, t]",0,[],9624289,2709,11366330
1,ext-journal-2020.csv,28172398,"[f, t]",0,[],12914371,5782,15252245
2,ext-journal-2021.csv,53698948,"[f, t]",0,[],36057607,18563,17622778
3,ext-journal-2022.csv,43483149,"[f, t]",0,[],33853235,22663,9607251
4,ext-journal-2023.csv,31937125,"[f, t]",0,[],23114518,24344,8798263
5,ext-journal-2024.csv,49023883,"[f, t]",0,[],39061884,23569,9938430
6,ext-journal-2025.csv,55541797,"[f, t, тревожное]",1,[ид_события | ид_канала_данных | дата | время | тревожное | значение_датчика],45767599,23272,9750926
7,ext-journal-2026.csv,30695389,"[f, t]",0,[],24695254,10606,5989529
8,журнал_событий_пример.csv,169993,"[false, true]",0,[],160062,32,9899


Из этой таблицы видно три особенности формата. Разберем каждую отдельно.

### 2.1. Строка заголовка посреди файла

**Наблюдение.** Почти во всех строках дата и время имеют правильный формат. Исключение – файл 2025 года:
в нем ровно одна строка, где в столбце `дата` написано слово «дата», а в остальных столбцах – тоже названия
столбцов:

In [10]:
odd_rows = first_look.loc[first_look["строк_с_нераспознанной_датой"] > 0, ["файл", "примеры_таких_строк"]]
odd_rows.explode("примеры_таких_строк")

,файл,примеры_таких_строк
6,ext-journal-2025.csv,ид_события | ид_канала_данных | дата | время | тревожное | значение_датчика


**Объяснение.** Это повтор строки заголовка. Скорее всего, годовой файл склеен из нескольких частей
выгрузки, и у одной из частей заголовок не срезали. Событием такая строка не является.

**Решение.** Удаляем строки, у которых в столбце `дата` стоит слово «дата». При очистке отдельно считаем,
сколько таких строк нашлось в каждом файле, и отдельно – сколько строк не удалось разобрать как дату и время
по другим причинам. Если заголовок – единственная аномалия, вторая цифра должна быть нулевой во всех годах.

### 2.2. Два формата флага тревоги

**Наблюдение.** В годовых журналах флаг `тревожное` записан как `t` / `f`, а в примере за 2026-08-01 –
как `true` / `false` (значение «тревожное» в файле 2025 года – из той же строки заголовка).

**Решение.** Приводим флаг к логическому типу: `t` и `true` означают тревогу, `f` и `false` – ее отсутствие.
Сам флаг берем таким, каким его выставила СМВУ, и не пересчитываем по тексту состояния: именно этот флаг
видит диспетчер, и именно по нему дальше будем судить, что считалось тревогой.

### 2.3. Три вида значений датчика

**Наблюдение.** Столбец `значение_датчика` хранит данные разной природы. В большинстве строк это число –
показание газоанализатора или термодатчика. Во многих – текст состояния: «Норма», «Неисправен», «Включен».
И есть небольшая группа значений, похожих на дату и время. Посмотрим на примеры каждого вида в примере
журнала за один день:

In [11]:
v = sample_raw["значение_датчика"]
kind = pd.Series("текст", index=v.index)
kind[pd.to_numeric(v, errors="coerce").notna()] = "число"
kind[v.str.match(etl.DATETIME_VALUE_RE)] = "дата"
pd.DataFrame({
    "вид_значения": ["число", "текст", "дата"],
    "строк": [(kind == k).sum() for k in ["число", "текст", "дата"]],
    "частые_значения": [", ".join(v[kind == k].value_counts().head(8).index) for k in ["число", "текст", "дата"]],
})

,вид_значения,строк,частые_значения
0,число,160062,"0.01, 0.00, 0.02, 0.03, 0.04, 0.07, 0.06, 0.09"
1,текст,9899,"Обнаружено движение, Движения нет, Норма, Неопределен, Выключен, Включен, Не..."
2,дата,32,"01.01.1970 03:00:00, 01.01.1970 03:00:01, 01.08.2026 08:04:13, 01.08.2026 08..."


In [12]:
date_values = sample_raw[kind == "дата"].astype({"ид_канала_данных": int}).merge(
    channels[["ид_канала_данных", "тип_датчика"]], on="ид_канала_данных", how="left")
date_values["тип_датчика"].value_counts(dropna=False).rename("строк").to_frame()

,строк
тип_датчика,
Состояние охраны,32


Значения-даты присылает один тип датчика – «Состояние охраны». Судя по значениям, это момент постановки
или снятия охраны, а `01.01.1970 03:00:00` (и соседние секунды) – нулевая отметка времени Unix
в московском поясе, то есть фактически пустое значение.

**Решение.** Разделяем записи на два класса (`класс_записи`):
- `измерение` – значение число (кладем в `числовое_значение`) или дата «Состояния охраны»;
- `состояние` – значение текст (кладем в `состояние`).

Отнести даты к измерениям, а не к состояниям, кажется произвольным решением. Обоснование появится ниже,
в разделе про идентификаторы событий: оказывается, у измерений и состояний разная нумерация, и записи с датами
нумеруются вместе с измерениями.

### 2.4. Полные дубли строк

Отдельно проверяем, нет ли в файлах строк, повторяющихся целиком – по всем шести столбцам сразу.
Для примера берем первый файл, 2019 года.

In [13]:
first_year = YEARS[0]
dup_example = (etl.read_raw(spark, JOURNALS[first_year])
               .groupBy(*etl.RAW_COLUMNS).count().filter("count > 1")
               .withColumnRenamed("count", "повторов")
               .limit(3).toPandas())
dup_example

,ид_события,ид_канала_данных,дата,время,тревожное,значение_датчика,повторов
0,1510213736,143509,2019-07-05,09:51:26,f,Неопределен,2
1,1510213941,143510,2019-07-05,09:51:32,f,Норма,2
2,1510245946,143541,2019-07-05,10:16:20,t,Не замкнут,2


**Наблюдение.** Такие строки есть: одно и то же событие того же канала, в ту же секунду, с тем же
идентификатором и значением записано дважды.

**Гипотеза.** Это дефект выгрузки: какой-то отрезок журнала попал в файл повторно. Если так, дубли должны
быть сосредоточены в коротком интервале времени, а не рассыпаны по всему году, как было бы при настоящих
повторных срабатываниях датчиков.

**Решение.** Удаляем полные дубли, оставляя одну копию. Проверку гипотезы о коротком интервале делаем после
очистки, когда будет известно, в каких файлах дубли есть.

Дедуплицируем только по всей строке целиком, а не по `ид_события`. Уже в примере за один день видно, что
с идентификатором не все просто (подробно – в разделе 5), и удаление по нему выбросило бы настоящие события.

## 3. Очистка

Все найденные правила собраны в функции `etl.clean_journal`. Она обрабатывает один годовой файл:
1. читает CSV строками и считает строки файла и строки-заголовки;
2. удаляет заголовки и полные дубли;
3. приводит типы: `ид_события` – целое число, `ид_канала_данных` – целое, `дата` и `время` склеиваются в
   `дата_время`, флаг `тревожное` – логический;
4. разбирает `значение_датчика` на `класс_записи`, `числовое_значение` и `состояние` (исходная строка
   значения тоже сохраняется);
5. отбрасывает строки, где дату и время не удалось разобрать, и считает их;
6. считает тревожные строки и строки от каналов, которых нет в справочнике.

Результат пишется в Parquet с разбиением по году и месяцу события (`год`, `месяц`), чтобы дальнейшие запросы
за период читали только нужные куски. Файлы обрабатываем по одному году, чтобы уложиться в память.

Запись идет в режиме дописывания: если событие из файла одного года датировано соседним годом, оно должно
лечь в партицию своего года и не затереть уже записанное. Поэтому перед полной пересборкой каталог хранилища
удаляется целиком, иначе повторный запуск удвоил бы данные.

In [14]:
shutil.rmtree(EVENTS_DIR, ignore_errors=True)

known_channels = set(channels["ид_канала_данных"].astype(int))
reports = []
for year, path in JOURNALS.items():
    t0 = time.perf_counter()
    df, rep = etl.clean_journal(spark, path, known_channels)
    etl.write_events(df, EVENTS_DIR)
    spark.catalog.clearCache()
    sec = time.perf_counter() - t0
    reports.append({"год_файла": year, **rep, "секунд": round(sec)})
    print(year, rep, f"{sec:.0f} с", flush=True)

report = pd.DataFrame(reports)
report.to_parquet(PROCESSED_DIR / "etl_report.parquet", index=False)
report

2019 {'строк_в_файле': 20993328, 'строк_заголовка': 0, 'полных_дублей': 1491, 'битых_дат': 0, 'строк_после_очистки': 20991837, 'тревожных': 134132, 'строк_каналов_вне_справочника': 2481504} 97 с


2020 {'строк_в_файле': 28172398, 'строк_заголовка': 0, 'полных_дублей': 0, 'битых_дат': 0, 'строк_после_очистки': 28172398, 'тревожных': 325662, 'строк_каналов_вне_справочника': 3278453} 143 с


2021 {'строк_в_файле': 53698948, 'строк_заголовка': 0, 'полных_дублей': 0, 'битых_дат': 0, 'строк_после_очистки': 53698948, 'тревожных': 1199639, 'строк_каналов_вне_справочника': 3323693} 241 с


2022 {'строк_в_файле': 43483149, 'строк_заголовка': 0, 'полных_дублей': 0, 'битых_дат': 0, 'строк_после_очистки': 43483149, 'тревожных': 133504, 'строк_каналов_вне_справочника': 328990} 211 с


2023 {'строк_в_файле': 31937125, 'строк_заголовка': 0, 'полных_дублей': 562449, 'битых_дат': 0, 'строк_после_очистки': 31374676, 'тревожных': 106945, 'строк_каналов_вне_справочника': 21106} 153 с


2024 {'строк_в_файле': 49023883, 'строк_заголовка': 0, 'полных_дублей': 0, 'битых_дат': 0, 'строк_после_очистки': 49023883, 'тревожных': 115203, 'строк_каналов_вне_справочника': 5257} 224 с


2025 {'строк_в_файле': 55541797, 'строк_заголовка': 1, 'полных_дублей': 0, 'битых_дат': 0, 'строк_после_очистки': 55541796, 'тревожных': 256471, 'строк_каналов_вне_справочника': 530} 263 с


2026 {'строк_в_файле': 30695389, 'строк_заголовка': 0, 'полных_дублей': 0, 'битых_дат': 0, 'строк_после_очистки': 30695389, 'тревожных': 195422, 'строк_каналов_вне_справочника': 122} 154 с


,год_файла,строк_в_файле,строк_заголовка,полных_дублей,битых_дат,строк_после_очистки,тревожных,строк_каналов_вне_справочника,секунд
0,2019,20993328,0,1491,0,20991837,134132,2481504,97
1,2020,28172398,0,0,0,28172398,325662,3278453,143
2,2021,53698948,0,0,0,53698948,1199639,3323693,241
3,2022,43483149,0,0,0,43483149,133504,328990,211
4,2023,31937125,0,562449,0,31374676,106945,21106,153
5,2024,49023883,0,0,0,49023883,115203,5257,224
6,2025,55541797,1,0,0,55541796,256471,530,263
7,2026,30695389,0,0,0,30695389,195422,122,154


## 4. Проверка результата очистки

### 4.1. Все строки учтены

Каждая строка исходного файла должна оказаться ровно в одной из групп: заголовок, лишняя копия дубля,
строка с неразобранной датой или чистое событие. Проверяем, что сумма групп равна числу строк файла.

In [15]:
report["сумма_групп"] = report[["строк_заголовка", "полных_дублей", "битых_дат", "строк_после_очистки"]].sum(axis=1)
assert (report["сумма_групп"] == report["строк_в_файле"]).all()
assert (report["битых_дат"] == 0).all()
report[["год_файла", "строк_в_файле", "строк_заголовка", "полных_дублей", "битых_дат", "строк_после_очистки",
        "сумма_групп"]]

,год_файла,строк_в_файле,строк_заголовка,полных_дублей,битых_дат,строк_после_очистки,сумма_групп
0,2019,20993328,0,1491,0,20991837,20993328
1,2020,28172398,0,0,0,28172398,28172398
2,2021,53698948,0,0,0,53698948,53698948
3,2022,43483149,0,0,0,43483149,43483149
4,2023,31937125,0,562449,0,31374676,31937125
5,2024,49023883,0,0,0,49023883,49023883
6,2025,55541797,1,0,0,55541796,55541797
7,2026,30695389,0,0,0,30695389,30695389


Все строки учтены. Строка заголовка нашлась только одна – в файле 2025 года. Строк с неразобранной датой
нет ни в одном файле, значит заголовок был единственной аномалией даты и времени. Полные дубли есть только
в двух файлах: 1 491 в 2019 году и 562 449 в 2023 году.

### 4.2. Где лежат дубли

Проверяем гипотезу о повторно выгруженном отрезке: смотрим, на какие дни приходятся дубли в тех файлах,
где они есть.

In [16]:
dup_days = []
for year in report.loc[report["полных_дублей"] > 0, "год_файла"]:
    dup_days.append(
        etl.read_raw(spark, JOURNALS[year])
        .groupBy(*etl.RAW_COLUMNS).count().filter("count > 1")
        .groupBy("дата").agg(F.sum(F.col("count") - 1).alias("лишних_копий"),
                             F.countDistinct("ид_канала_данных").alias("каналов"))
        .withColumn("год_файла", F.lit(year))
        .toPandas())
dup_days = pd.concat(dup_days, ignore_index=True).sort_values(["год_файла", "дата"]) if dup_days else pd.DataFrame()
dup_days

,дата,лишних_копий,каналов,год_файла
0,2019-07-05,1491,206,2019
4,2023-11-24,1,1,2023
3,2023-11-25,62488,367,2023
5,2023-11-26,99714,595,2023
2,2023-11-27,146977,1183,2023
6,2023-11-28,133457,1275,2023
1,2023-11-29,119812,812,2023


Гипотеза подтверждается. В 2019 году все 1 491 лишняя копия относится к одному дню, 2019-07-05 (206 каналов).
В 2023 году 562 449 копий лежат в интервале 2023-11-24 … 2023-11-29: на 24 ноября приходится лишь одна копия,
основная масса – на 25-29 ноября, от 62 до 147 тыс. в день и от 367 до 1 275 каналов. За весь остальной год
в обоих файлах нет ни одного дубля. Такая концентрация похожа на повторную выгрузку куска журнала, а не
на поведение датчиков, так что удаление копий ничего не теряет, а без него счетчики событий за эти дни
были бы завышены на десятки тысяч событий в сутки.

### 4.3. Каждый файл лежит в своем году

Годовые файлы названы по году, но это еще не гарантирует, что внутри только события этого года. Читаем
хранилище и сравниваем число событий в каждой годовой партиции с числом чистых строк соответствующего файла.

In [17]:
events = spark.read.parquet(str(EVENTS_DIR))
by_year = (events.groupBy("год").agg(F.count("*").alias("строк_в_хранилище"),
                                    F.min("дата_время").alias("первое_событие"),
                                    F.max("дата_время").alias("последнее_событие"))
           .orderBy("год").toPandas())
landed = by_year.merge(report[["год_файла", "строк_после_очистки"]], left_on="год", right_on="год_файла",
                       how="outer")
landed["разница"] = landed["строк_в_хранилище"] - landed["строк_после_очистки"]
assert (landed["разница"] == 0).all()
landed.drop(columns="год_файла")

,год,строк_в_хранилище,первое_событие,последнее_событие,строк_после_очистки,разница
0,2019,20991837,2019-01-01 00:00:21,2019-12-31 23:59:52,20991837,0
1,2020,28172398,2020-01-01 00:00:04,2020-12-31 23:59:59,28172398,0
2,2021,53698948,2021-01-01 00:00:01,2021-12-31 23:59:59,53698948,0
3,2022,43483149,2022-01-01 00:00:00,2022-12-31 23:59:56,43483149,0
4,2023,31374676,2023-01-01 00:00:02,2023-12-31 23:59:58,31374676,0
5,2024,49023883,2024-01-01 00:00:00,2024-12-31 23:59:59,49023883,0
6,2025,55541796,2025-01-01 00:00:05,2025-12-31 23:59:59,55541796,0
7,2026,30695389,2026-01-01 00:00:00,2026-06-30 23:59:59,30695389,0


Разница нулевая во всех годах: каждый файл содержит события только своего года. Журнал начинается
2019-01-01 00:00:21 и заканчивается 2026-06-30 23:59:59.

### 4.4. Пропущенные дни

Непрерывный ли журнал? Если в какие-то дни нет ни одного события, это важно знать заранее: в такие дни
нельзя ни обучать модель, ни судить об отсутствии инцидентов.

In [18]:
days = set(events.select(F.to_date("дата_время").alias("день")).distinct().toPandas()["день"])
calendar = pd.date_range(min(days), max(days), freq="D").date
missing_days = sorted(set(calendar) - days)
print("дней в периоде:", len(calendar), "| дней без единого события:", len(missing_days))
pd.DataFrame({"день_без_событий": missing_days})

дней в периоде: 2738 | дней без единого события: 3


,день_без_событий
0,2024-04-07
1,2024-04-08
2,2026-06-01


Из 2 738 дней периода событий нет только в трех: 2024-04-07, 2024-04-08 и 2026-06-01. Для системы,
которая присылает тысячи сообщений в сутки, полностью пустой день означает не «ничего не случилось», а
отсутствие данных: сбой сбора или выгрузки. Причину по журналу установить нельзя; при построении выборки
эти дни помечаем как пропуск данных, а не как спокойные.

## 5. Идентификатор события не уникален

**Наблюдение.** Естественно ожидать, что `ид_события` – уникальный номер записи. Проверим это на очищенном
хранилище: сколько значений `ид_события` встречается больше одного раза.

In [19]:
dup_ids = (events.groupBy("ид_события").agg(
               F.count("*").alias("записей"),
               F.countDistinct("класс_записи").alias("классов_записи"),
               F.array_sort(F.collect_set("год")).alias("годы"),
               F.min_by("класс_записи", "дата_время").alias("первый_класс"))
           .filter(F.col("записей") > 1).cache())
print("повторяющихся ид_события:", dup_ids.count())
dup_ids.groupBy("записей", "классов_записи").count().withColumnRenamed("count", "идентификаторов") \
    .orderBy("записей", "классов_записи").toPandas()

повторяющихся ид_события: 5462138


,записей,классов_записи,идентификаторов
0,2,2,5462138


Повторы есть, и много: 5 462 138 идентификаторов встречаются ровно дважды, больше двух раз – ни один. Вот как
выглядит одна такая пара целиком:

In [20]:
pair_ids = [r["ид_события"] for r in dup_ids.orderBy("ид_события").limit(1).collect()]
(events.filter(F.col("ид_события").isin(pair_ids))
 .select("ид_события", "ид_канала_данных", "дата_время", "класс_записи", "значение_датчика", "тревожное")
 .orderBy("ид_события", "дата_время").toPandas())

,ид_события,ид_канала_данных,дата_время,класс_записи,значение_датчика,тревожное
0,1385057977,77537,2019-01-01 07:22:25,состояние,Норма,False
1,1385057977,196764,2020-07-31 06:37:44,измерение,0.00,False


Записи с одним номером здесь относятся к разным каналам, разному времени и разному смыслу: одна – числовой замер,
другая – смена состояния. Это не дубль одного события.

**Гипотеза.** В СМВУ измерения и состояния хранятся в двух разных таблицах, у каждой свой счетчик
идентификаторов, а журнал – их объединение. Тогда номера из двух счетчиков время от времени совпадают.

Подсказка в пользу гипотезы есть уже в примере за один день: номера текстовых записей и номера числовых
лежат в совершенно разных диапазонах, а записи со значением-датой – внутри диапазона чисел.

In [21]:
sample_ids = sample_raw.assign(вид_значения=kind, ид=sample_raw["ид_события"].astype("int64"))
sample_ids.groupby("вид_значения")["ид"].agg(записей="size", минимальный_ид="min", максимальный_ид="max")

,записей,минимальный_ид,максимальный_ид
вид_значения,,,
дата,32,4524501739,4525243744
текст,9899,3441201782,3441631016
число,160062,4524058382,4525472879


За один день два счетчика не пересекаются, но за восемь лет один счетчик может «догнать» другой, и тогда
номера совпадут. Если гипотеза верна, должны выполняться три условия:
1. в каждой паре с одинаковым `ид_события` одна запись – измерение, другая – состояние (два измерения или
   два состояния с одним номером опровергли бы гипотезу);
2. пара (`ид_события`, `класс_записи`) уникальна, то есть внутри своего класса номер – настоящий ключ;
3. совпадения возникают там, где один счетчик проходит значения, которые другой проходил раньше, поэтому
   записи пары могут относиться к разным годам, а класс, который встречается в паре первым, со временем
   должен смениться.

Условие 1 видно из таблицы выше: все 100 % пар состоят из измерения и
состояния. Проверяем условие 2.

In [22]:
key_repeats = events.groupBy("ид_события", "класс_записи").count().filter("count > 1").count()
print("повторов ключа (ид_события, класс_записи):", key_repeats)

повторов ключа (ид_события, класс_записи): 0


Повторов ключа нет ни одного: внутри класса номер уникален.
Проверяем условие 3: между какими годами возникают пары и какой класс в паре появился раньше.

In [23]:
(dup_ids.groupBy("годы", "первый_класс").count().withColumnRenamed("count", "пар")
 .withColumn("годы", F.array_join("годы", " и "))
 .toPandas()
 .pivot(index="годы", columns="первый_класс", values="пар").fillna(0).astype(int))

первый_класс,измерение,состояние
годы,,
2019 и 2020,0,323028
2019 и 2021,0,227241
2020 и 2021,0,1065634
2021,0,775099
2021 и 2022,0,642232
2022,186025,481292
2022 и 2023,133723,0
2023,226852,0
2023 и 2024,390767,0


In [24]:
dup_ids.unpersist()

DataFrame[ид_события: bigint, записей: bigint, классов_записи: bigint, годы: array<int>, первый_класс: string]

**Результат.** Пары возникают во всех годах. Часть из них лежит внутри одного года (2021, 2022, 2023), но
многие связывают разные годы: 2019 и 2020 (323 тыс. пар), 2020 и 2021 (1,07 млн), 2019 и 2021, 2023 и 2025,
2024 и 2026 (304 тыс.). Порядок классов в паре меняется ровно один раз. Во всех парах, которые связывают годы
2019-2022, первой по времени идет запись состояния. Внутри 2022 года встречаются оба варианта (481 тыс. пар,
где раньше состояние, и 186 тыс., где раньше измерение). Начиная с пар 2022 и 2023 года и во всех более
поздних первым всегда идет измерение.

Это ровно та картина, которую предсказывает гипотеза двух счетчиков. До 2022 года счетчик измерений отставал
и выдавал номера, которые счетчик состояний уже использовал раньше. С 2021 года замеров в журнале в два-три
раза больше, чем смен состояний (в 2019-2020 годах было наоборот, это видно в таблице первого осмотра),
поэтому счетчик измерений рос быстрее, в 2022 году догнал счетчик состояний и ушел вперед. После этого уже
счетчик состояний повторяет номера, которые измерения прошли раньше. Пример за 2026-08-01 с этим согласуется:
номера измерений там около 4,52 млрд, а номера состояний около 3,44 млрд.

**Вывод.** Гипотеза о двух независимых счетчиках подтверждается всеми тремя проверками. Для дальнейшей
работы это значит:
- `ид_события` не ключ записи; ключ – пара (`ид_события`, `класс_записи`);
- дедуплицировать по `ид_события` нельзя (мы и не делали: дубли удалялись только по полной строке);
- сортировать события по `ид_события` между классами бессмысленно. Порядок событий задает `дата_время`, а
  `ид_события` годится только как уточнение порядка внутри одного класса, когда у канала несколько записей
  в одну секунду;
- записи с датами в значении нумеруются вместе с измерениями, поэтому отнесены к классу «измерение».

## 6. События от каналов, которых нет в справочнике

При очистке мы считали строки, у которых `ид_канала_данных` не найден в справочнике каналов. Для таких
событий неизвестны тип датчика и объект. Посмотрим, сколько их по годам и сколько за ними разных каналов.

In [25]:
unknown_events = events.join(spark.createDataFrame(channels[["ид_канала_данных"]]), "ид_канала_данных",
                             "left_anti")
print("разных каналов вне справочника за все годы:",
      unknown_events.select("ид_канала_данных").distinct().count())
unknown_channels = (unknown_events.groupBy("год")
                    .agg(F.countDistinct("ид_канала_данных").alias("каналов_вне_справочника"))
                    .toPandas())
unknown = report[["год_файла", "строк_после_очистки", "строк_каналов_вне_справочника"]].merge(
    unknown_channels, left_on="год_файла", right_on="год", how="left").drop(columns="год")
unknown["каналов_вне_справочника"] = unknown["каналов_вне_справочника"].fillna(0).astype(int)
unknown["доля_строк_вне_справочника_%"] = (
    100 * unknown["строк_каналов_вне_справочника"] / unknown["строк_после_очистки"]).round(2)
unknown

разных каналов вне справочника за все годы: 1142


,год_файла,строк_после_очистки,строк_каналов_вне_справочника,каналов_вне_справочника,доля_строк_вне_справочника_%
0,2019,20991837,2481504,1004,11.82
1,2020,28172398,3278453,990,11.64
2,2021,53698948,3323693,829,6.19
3,2022,43483149,328990,558,0.76
4,2023,31374676,21106,106,0.07
5,2024,49023883,5257,30,0.01
6,2025,55541796,530,12,0.00
7,2026,30695389,122,2,0.00


**Наблюдение.** Доля таких событий сильно зависит от года: 11,8 % строк в 2019 году, 11,6 % в 2020,
6,2 % в 2021, 0,76 % в 2022, 0,07 % в 2023 и практически ноль в 2024-2026 (от 5 тыс. строк в 2024 до 122 строк
в 2026). Вместе с долей строк падает и число таких каналов: около тысячи в 2019-2020, 829 в 2021, 558 в 2022,
106 в 2023 и единицы-десятки в последние годы. Всего это 9 439 655 строк от 1 142 разных каналов.
В ранних годах это заметная часть журнала, в последних – единичные строки, а спад приходится на 2021-2022 годы.

**Что это значит сейчас.** Справочник каналов выглядит как снимок текущей конфигурации системы, а часть
каналов, которые работали раньше, в него не попала. Почему так – разберем при исследовании журнала. На этапе
предобработки эти события не удаляем: они настоящие, а их поведение само по себе может подсказать, что
происходило с системой. При соединении со справочником используем левое соединение, чтобы их не потерять,
и помним, что для ранних лет картина по типам датчиков и объектам будет неполной.

## 7. Пример за 2026-08-01 тем же конвейером

Пример журнала за один день пропускаем через ту же функцию очистки. Он лежит вне периода годовых файлов и
в обучении не участвует; это «живой» день, на котором будем показывать работу сервиса. Важно, что он
проходит тот же конвейер: так формат `true` / `false` его флага гарантированно понимается так же, как
`t` / `f` в годовых журналах.

In [26]:
sample, sample_report = etl.clean_journal(spark, SAMPLE, known_channels)
sample.write.mode("overwrite").parquet(str(PROCESSED_DIR / "sample"))
spark.catalog.clearCache()
pd.Series(sample_report, name="пример_2026_08_01").to_frame()

,пример_2026_08_01
строк_в_файле,169993
строк_заголовка,0
полных_дублей,0
битых_дат,0
строк_после_очистки,169993
тревожных,207
строк_каналов_вне_справочника,0


## Что получилось

Журнал очищен и сохранен в `datasets/processed/events` в формате Parquet с разбиением по `год` / `месяц`.
Рядом лежат справочник каналов `channels.parquet`, отчет об очистке `etl_report.parquet` и пример дня
в `sample/`. Состав хранилища:

In [27]:
meaning = {
    "ид_события": "номер записи из источника; не уникален – у измерений и состояний свои счетчики",
    "ид_канала_данных": "канал датчика, ссылка на справочник каналов",
    "дата_время": "момент события, московское время (склейка исходных `дата` и `время`)",
    "тревожное": "флаг тревоги от СМВУ, приведен к логическому типу из t/f и true/false",
    "значение_датчика": "исходное значение строкой, как в CSV",
    "класс_записи": "«измерение» (число или дата) или «состояние» (текст)",
    "числовое_значение": "значение измерения числом, у состояний пусто",
    "состояние": "текст состояния, у измерений пусто",
    "год": "год события (партиция)",
    "месяц": "месяц события (партиция)",
}
pd.DataFrame([{"столбец": f.name, "тип": f.dataType.simpleString(), "смысл": meaning.get(f.name, "")}
              for f in events.schema.fields])

,столбец,тип,смысл
0,ид_события,bigint,номер записи из источника; не уникален – у измерений и состояний свои счетчики
1,ид_канала_данных,int,"канал датчика, ссылка на справочник каналов"
2,дата_время,timestamp,"момент события, московское время (склейка исходных `дата` и `время`)"
3,тревожное,boolean,"флаг тревоги от СМВУ, приведен к логическому типу из t/f и true/false"
4,значение_датчика,string,"исходное значение строкой, как в CSV"
5,класс_записи,string,«измерение» (число или дата) или «состояние» (текст)
6,числовое_значение,double,"значение измерения числом, у состояний пусто"
7,состояние,string,"текст состояния, у измерений пусто"
8,год,int,год события (партиция)
9,месяц,int,месяц события (партиция)


In [28]:
totals = pd.DataFrame({
    "показатель": ["строк в исходных файлах", "удалено строк заголовка", "удалено полных дублей",
                   "строк с неразобранной датой", "событий в хранилище", "из них тревожных",
                   "из них от каналов вне справочника"],
    "значение": [report["строк_в_файле"].sum(), report["строк_заголовка"].sum(), report["полных_дублей"].sum(),
                 report["битых_дат"].sum(), report["строк_после_очистки"].sum(), report["тревожных"].sum(),
                 report["строк_каналов_вне_справочника"].sum()],
})
totals

,показатель,значение
0,строк в исходных файлах,313546017
1,удалено строк заголовка,1
2,удалено полных дублей,563940
3,строк с неразобранной датой,0
4,событий в хранилище,312982076
5,из них тревожных,2466978
6,из них от каналов вне справочника,9439655


Коротко о том, что сделано и почему:

| Шаг | Что увидели в данных | Что сделали | Строк |
|---|---|---|---:|
| Заголовок внутри файла | строка из названий столбцов посреди файла 2025 года | удалили | 1 |
| Полные дубли | повтор выгрузки: 2019-07-05 и 2023-11-24 … 29, каждая строка дважды | оставили одну копию | 563 940 |
| Флаг тревоги | `t` / `f` в годовых журналах, `true` / `false` в примере дня | привели к логическому типу, по тексту не пересчитывали | все |
| Значение датчика | число, текст состояния или дата | разделили на `класс_записи`, `числовое_значение`, `состояние` | все |
| `ид_события` | повторяется: два независимых счетчика у измерений и состояний | не используем как ключ; ключ – с `класс_записи` | 5 462 138 пар |
| Каналы вне справочника | много в ранних годах, почти нет в последних | оставили, соединение со справочником левое | 9 439 655 |
| Пропущенные дни | 2024-04-07, 2024-04-08, 2026-06-01 – ни одного события | пометить как пропуск данных | – |

Итог: в хранилище **312 982 076** событий с 2019-01-01 00:00:21 по 2026-06-30 23:59:59,
из них **2 466 978** тревожных (0,79 %). Каждая строка исходных файлов учтена,
события каждого файла лежат в своем году, дат, которые не удалось разобрать, нет.

Открытыми остаются два вопроса, которые требуют уже исследования содержания журнала, а не формата: откуда
в ранних годах каналы, которых нет в справочнике, и как на журнале отразилось внедрение новой СМВУ в 2021
году, о котором говорит заказчик.